# Bembidion scenario preparation

This notebook illustrates how typical land use / land cover (LULC) data can be transformed into landscapes for bembidion
simulations. The main task of the transformation is to map LULC types to the habitat types required by the bembidion
model and to provide a representation of the landscape that can be easily digested by the bembidion model.

This example uses a small agricultural landscape in North Rhine-Westfalia.

![NRW2 scenario over satellite imagery](data/nrw2.png "NRW2")

The landscape has an extent of approximately 2 km x 2 km, is represented in a shapefile (see `data/LULC.shp`) and shows
the LULC types of 378 distinct features.

A look into the attribute table of the shapefile reveals eight different LULC types in use. A comparison with satellite
imagery allows to approximate the LULC types as follows:

- `10`: agricultural fields
- `20`: shrubs
- `30`: forest
- `53`: aquatic vegetation strips
- `60`: water
- `90`: buildings
- `91`: roads
- `92`: road vegetation strips


## Processing

### 1. Rasterization

The world of the bembidion is currently a raster world. This could change in the future, but currently it's living in a
raster world with a fixed cell size of 1 m x 1 m. For this reason, the vector layer should be converted into a raster
layer with according resolution, where each cell value represents its LULC type. This can be done, e.g., in a GIS,
using the GDAL CLI (command below) or in Python.

```
gdal_rasterize -l LULC -a LULCtype_1 -tr 1.0 1.0 -ot UInt16 -of GTiff -co COMPRESS=DEFLATE -co PREDICTOR=2
-co ZLEVEL=9 F:\BembidionDev\data\LULC.shp F:/BembidionDev/data/nrw2.tif
```

The rasterized GeoTiff can also be found in the `data` folder.


### 2. GeoTiff to NumPy array

The GeoTiff can be load into a NumPy array, e.g., using `geotiff`. Having the data as a NumPy array is a good starting
point for further data transformations.

It is important that we keep the geodata in a projected CRS that uses meters as unit and that we pass the EPSG code of
this CRS (here, 25832) to the constructor of the GeoTiff object.

Missing values (cells not represented in the original shapefile) are reported with a LULC type of `0`. For the bembidion
model to work properly, `0`-vlaues should not occur inside the landscape, only outside of it. That is, it has to be
ensured that the shapefile is compact, i.e., without inner holes, and the compactness is kept during rasterization.

In [24]:
import geotiff
import numpy
raster = geotiff.GeoTiff("data/nrw2.tif", as_crs=25832)
lulc = numpy.array(raster.read())
lulc

array([[ 0,  0,  0, ...,  0,  0,  0],
       [ 0, 10, 10, ..., 20,  0,  0],
       [ 0, 10, 10, ..., 20,  0,  0],
       ...,
       [ 0,  0,  0, ..., 10, 10,  0],
       [ 0,  0,  0, ..., 10, 10,  0],
       [ 0,  0,  0, ..., 10, 10,  0]], shape=(2007, 2003), dtype=uint16)

### 3. Raster cell coordinates
The `geotiff` package contains a convinient function to extract longitudinal and latitudinal coordinates of the raster
cells in the same shape in which we extracted the LULC types.

In [26]:
longitude_coords, latitude_coords = raster.get_coord_arrays()
longitude_coords

array([[319876.2044, 319877.2044, 319878.2044, ..., 321876.2044,
        321877.2044, 321878.2044],
       [319876.2044, 319877.2044, 319878.2044, ..., 321876.2044,
        321877.2044, 321878.2044],
       [319876.2044, 319877.2044, 319878.2044, ..., 321876.2044,
        321877.2044, 321878.2044],
       ...,
       [319876.2044, 319877.2044, 319878.2044, ..., 321876.2044,
        321877.2044, 321878.2044],
       [319876.2044, 319877.2044, 319878.2044, ..., 321876.2044,
        321877.2044, 321878.2044],
       [319876.2044, 319877.2044, 319878.2044, ..., 321876.2044,
        321877.2044, 321878.2044]], shape=(2007, 2003))

### 4. NumPy to Polars
With the LULC type and coordinate arrays prepared, we can compile a serial representation in a tabular format.

In [34]:
import polars as pl
lulc = pl.DataFrame({"x": longitude_coords.flatten(), "y": latitude_coords.flatten(), "lulc": lulc.flatten()})
lulc

x,y,lulc
f64,f64,u16
319876.2044,5.7043e6,0
319877.2044,5.7043e6,0
319878.2044,5.7043e6,0
319879.2044,5.7043e6,0
319880.2044,5.7043e6,0
…,…,…
321874.2044,5.7023e6,10
321875.2044,5.7023e6,10
321876.2044,5.7023e6,10


### 5. Cleanup, accessibility mapping and saving

The cleanup steps comprise removal of cells without LULC type (value = `0`) and expressing x- and y-coordinates in full
meters, i.e., as integers.

Accessibility mapping has to be done by expert judgment and is a translation of LULC types into the accessibility of a
cell for the bembidions. The current model distinguishes only three kinds of accessibility: full, partial and none. As
the LULC type codes may differ across scenarios, it is a good idea to leave the mapping and the judgement within the
scenario generation process.

In this scenario, we consider roads as accessible, and only forests, water and buildings as inaccesible.

In [52]:
lulc = (
    lulc
    .filter(pl.col("lulc") > 0)
    .select(
        pl.col("x").round().cast(pl.Int32),
        pl.col("y").round().cast(pl.Int32),
        accessibility=pl
        .when(pl.col("lulc").is_in((10, 20, 53, 92))).then(pl.lit("full"))
        .when(pl.col("lulc").is_in((30, 60, 90))).then(pl.lit("none"))
        .when(lulc=91).then(pl.lit("partial"))
    )
)
lulc.write_parquet("data/lulc.parquet")
lulc

x,y,accessibility
i32,i32,str
319877,5704301,"""full"""
319878,5704301,"""full"""
319879,5704301,"""full"""
319880,5704301,"""full"""
319881,5704301,"""full"""
…,…,…
321873,5702296,"""full"""
321874,5702296,"""full"""
321875,5702296,"""full"""
